In [1]:
!pip install -q pypdf python-docx pandas sentence-transformers chromadb groq gradio

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


In [3]:
import os
import re
import pandas as pd
from pypdf import PdfReader
from docx import Document


In [6]:
from google.colab import files

os.makedirs("documents", exist_ok=True)

uploaded = files.upload()

for file_name in uploaded:
    with open("documents/" + file_name, "wb") as f:
        f.write(uploaded[file_name])

print("Uploaded files:", os.listdir("documents"))

Saving machine_learning.csv to machine_learning.csv
Uploaded files: ['linear_algebra_course_guide.txt', 'machine_learning.csv', 'python_programming_course_guide.docx', 'strang_linear_algebra_A_CR.pdf.pdf']


In [7]:
documents = []

for file_name in os.listdir("documents"):
    path = "documents/" + file_name

    if file_name.endswith(".pdf"):
        reader = PdfReader(path)
        for i, page in enumerate(reader.pages):
            text = page.extract_text()
            if text:
                documents.append({"source": file_name, "location": "page " + str(i + 1), "text": text})

    elif file_name.endswith(".docx"):
        doc = Document(path)
        text = "\n".join([p.text for p in doc.paragraphs])
        documents.append({"source": file_name, "location": "full file", "text": text})

    elif file_name.endswith(".csv"):
        df = pd.read_csv(path)
        for i, row in df.iterrows():
            text = " | ".join([str(value) for value in row.values])
            documents.append({"source": file_name, "location": "row " + str(i + 1), "text": text})

    elif file_name.endswith(".txt"):
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()
        documents.append({"source": file_name, "location": "full file", "text": text})

print("Number of extracted documents:", len(documents))
print("First document source:", documents[0]["source"])
print("Preview:", documents[0]["text"][:200])

Number of extracted documents: 76
First document source: linear_algebra_course_guide.txt
Preview: Linear Algebra Fundamentals & Core Concepts
Course: Linear Algebra | Source: MIT OpenCourseWare (Gilbert Strang - Introduction to Linear Algebra) & GeeksforGeeks (https://ocw.mit.edu/courses/18-06-lin


In [8]:
def clean_text(text):
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

for doc in documents:
    doc["text"] = clean_text(doc["text"])

documents = [doc for doc in documents if len(doc["text"]) > 20]

print("Number of documents after cleaning:", len(documents))
print("Preview:", documents[0]["text"][:200])

Number of documents after cleaning: 76
Preview: Linear Algebra Fundamentals & Core Concepts Course: Linear Algebra | Source: MIT OpenCourseWare (Gilbert Strang - Introduction to Linear Algebra) & GeeksforGeeks (https://ocw.mit.edu/courses/18-06-lin


In [9]:
chunk_size = 500
overlap = 100

chunks = []

for doc in documents:
    text = doc["text"]
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk_text = text[start:end]
        chunks.append({"source": doc["source"], "location": doc["location"], "text": chunk_text})
        start = start + chunk_size - overlap

print("Number of chunks:", len(chunks))
print("Example chunk:", chunks[0]["text"])
print("Its source:", chunks[0]["source"])

Number of chunks: 205
Example chunk: Linear Algebra Fundamentals & Core Concepts Course: Linear Algebra | Source: MIT OpenCourseWare (Gilbert Strang - Introduction to Linear Algebra) & GeeksforGeeks (https://ocw.mit.edu/courses/18-06-linear-algebra-spring-2010/ & https://www.geeksforgeeks.org/linear-algebra/) ================================================================================ 1. Vectors, Vector Spaces, and Linear Combinations A vector is an ordered array of numbers representing a point or direction in an n-dimensional 
Its source: linear_algebra_course_guide.txt


In [10]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [chunk["text"] for chunk in chunks]
embeddings = embed_model.encode(texts, show_progress_bar=True)

print("Embeddings shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Embeddings shape: (205, 384)


In [11]:
import chromadb

client = chromadb.Client()
collection = client.get_or_create_collection(name="course_chunks", metadata={"hnsw:space": "cosine"})

ids = [str(i) for i in range(len(chunks))]
metadatas = [{"source": chunk["source"], "location": chunk["location"]} for chunk in chunks]

collection.add(
    ids=ids,
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

print("Chunks stored in vector DB:", collection.count())

Chunks stored in vector DB: 205


In [12]:
def retrieve(question, top_k=4):
    question_embedding = embed_model.encode([question]).tolist()
    results = collection.query(query_embeddings=question_embedding, n_results=top_k)

    retrieved = []
    for i in range(len(results["documents"][0])):
        retrieved.append({
            "text": results["documents"][0][i],
            "source": results["metadatas"][0][i]["source"],
            "location": results["metadatas"][0][i]["location"],
            "score": 1 - results["distances"][0][i]
        })
    return retrieved

test_results = retrieve("What is the LEGB scope rule in Python?")

for r in test_results:
    print("Score:", round(r["score"], 3), "| Source:", r["source"], "|", r["location"])
    print(r["text"][:150])
    print("-----")

Score: 0.667 | Source: python_programming_course_guide.docx | full file
ge/) 4. Functions, Scope (LEGB), and Closures Functions in Python are first-class objects, meaning they can be assigned to variables, passed as argume
-----
Score: 0.525 | Source: python_programming_course_guide.docx | full file
. - Comprehensions: Concise syntax for constructing sequences from iterables: * List Comprehension: [x**2 for x in range(10) if x % 2 == 0] * Dictiona
-----
Score: 0.482 | Source: python_programming_course_guide.docx | full file
 the order: Local -> Enclosing -> Global -> Built-in. - Lambda Functions: Anonymous single-expression functions written as lambda args: expression, co
-----
Score: 0.449 | Source: python_programming_course_guide.docx | full file
hods: Special double-underscore methods like __str__, __repr__, __len__, and __eq__ allow custom objects to integrate with built-in Python operations.
-----


In [15]:
from groq import Groq
from getpass import getpass

groq_api_key = getpass("Enter your Groq API key: ")
groq_client = Groq(api_key=groq_api_key)

def ask_llm(prompt):
    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content

print(ask_llm("Say hello in one short sentence."))

Enter your Groq API key: ··········
Hello!


In [16]:
def evaluate_chunk(question, chunk_text):
    prompt = f"""You are checking if a text chunk helps answer a question.

Question: {question}

Chunk: {chunk_text}

Does this chunk contain information that helps answer the question?
Answer with only one word: yes or no."""

    answer = ask_llm(prompt)
    return answer.strip().lower().startswith("yes")

def evaluate_chunks(question, retrieved):
    good_chunks = []
    for chunk in retrieved:
        if evaluate_chunk(question, chunk["text"]):
            good_chunks.append(chunk)
    return good_chunks

question = "What is the LEGB scope rule in Python?"
retrieved = retrieve(question)
good_chunks = evaluate_chunks(question, retrieved)

print("Retrieved chunks:", len(retrieved))
print("Good chunks after evaluation:", len(good_chunks))
for c in good_chunks:
    print("Score:", round(c["score"], 3), "| Source:", c["source"])

Retrieved chunks: 4
Good chunks after evaluation: 2
Score: 0.667 | Source: python_programming_course_guide.docx
Score: 0.482 | Source: python_programming_course_guide.docx


In [17]:
def rewrite_query(question):
    prompt = f"""The search for this question did not find good results in a course documents database.
Rewrite the question to make it clearer and easier to search.
Use different words and technical terms that may appear in course materials.

Original question: {question}

Return only the rewritten question, nothing else."""

    new_question = ask_llm(prompt)
    return new_question.strip()

old_question = "How do I stop my model from memorizing the training data?"
new_question = rewrite_query(old_question)

print("Old question:", old_question)
print("New question:", new_question)

Old question: How do I stop my model from memorizing the training data?
New question: What methods can I apply to prevent overfitting and ensure my model generalizes instead of simply memorizing the training dataset?


In [18]:
def generate_answer(question, good_chunks):
    context = ""
    for chunk in good_chunks:
        context = context + chunk["text"] + "\n\n"

    prompt = f"""Answer the question using only the context below.
If the context does not contain the answer, say exactly: I could not find the answer in the documents.
Do not use any outside knowledge.

Context:
{context}

Question: {question}

Answer:"""

    return ask_llm(prompt)

question = "What is the LEGB scope rule in Python?"
retrieved = retrieve(question)
good_chunks = evaluate_chunks(question, retrieved)
answer = generate_answer(question, good_chunks)

print("Question:", question)
print("Answer:", answer)
print("Sources:")
for chunk in good_chunks:
    print("-", chunk["source"], "|", chunk["location"])

Question: What is the LEGB scope rule in Python?
Answer: The LEGB scope rule states that Python resolves variable names in the following order: **Local → Enclosing → Global → Built‑in**.
Sources:
- python_programming_course_guide.docx | full file
- python_programming_course_guide.docx | full file


In [19]:
def corrective_rag(question):
    used_rewrite = False
    new_question = question

    #1: retrieve and evaluate
    retrieved = retrieve(question)
    good_chunks = evaluate_chunks(question, retrieved)

    #2: if context is weak, rewrite the question and try again
    if len(good_chunks) == 0:
        used_rewrite = True
        new_question = rewrite_query(question)
        retrieved = retrieve(new_question)
        good_chunks = evaluate_chunks(new_question, retrieved)

    #3: if still nothing, return the fallback answer
    if len(good_chunks) == 0:
        return {
            "answer": "I could not find the answer in the documents.",
            "sources": [],
            "used_rewrite": used_rewrite,
            "new_question": new_question
        }

    #4: generate the answer from good chunks only
    answer = generate_answer(question, good_chunks)

    sources = []
    for chunk in good_chunks:
        source = chunk["source"] + " | " + chunk["location"]
        if source not in sources:
            sources.append(source)

    return {
        "answer": answer,
        "sources": sources,
        "used_rewrite": used_rewrite,
        "new_question": new_question
    }

result = corrective_rag("How do I stop my model from memorizing the training data?")

print("Answer:", result["answer"])
print("Sources:", result["sources"])
print("Used rewrite:", result["used_rewrite"])
print("New question:", result["new_question"])

Answer: To prevent your model from simply memorizing the training data, you can apply several techniques mentioned in the context:

- **Regularization**: Use L1 (Lasso) or L2 (Ridge) regularization to add penalties to large weights, which reduces model complexity.
- **Cross‑Validation**: Implement k‑fold cross‑validation to verify that the model generalizes well to unseen data.
- **Dropout (for Neural Networks)**: Randomly drop neurons during training so the model doesn’t become overly dependent on specific nodes.
- **Simpler Models**: Choose a less complex model when a simpler one can adequately explain the data, avoiding unnecessary complexity that can lead to memorization.
Sources: ['machine_learning.csv | row 2']
Used rewrite: False
New question: How do I stop my model from memorizing the training data?


In [20]:
test_questions = [
    # Normal
    {"type": "normal", "question": "What is overfitting in machine learning and how can it be avoided?"},
    {"type": "normal", "question": "What is the LEGB scope rule in Python?"},
    {"type": "normal", "question": "What does A = CR mean in matrix multiplication?"},

    # Tricky
    {"type": "tricky", "question": "How do I stop my model from memorizing the training data?"},
    {"type": "tricky", "question": "In what order does Python look for a variable name?"},
    {"type": "tricky", "question": "How can I rebuild a matrix from only its independent columns?"},

    # Out of scope
    {"type": "out_of_scope", "question": "Who won the World Cup in 2018?"},
    {"type": "out_of_scope", "question": "What is the capital of France?"},
    {"type": "out_of_scope", "question": "How do I bake a chocolate cake?"},
]

for item in test_questions:
    result = corrective_rag(item["question"])

    print("Type:", item["type"])
    print("Question:", item["question"])
    print("Used rewrite:", result["used_rewrite"])
    print("Answer:", result["answer"][:200])
    print("Sources:", result["sources"])
    print("=" * 60)

Type: normal
Question: What is overfitting in machine learning and how can it be avoided?
Used rewrite: False
Answer: **Overfitting** occurs when a model learns not only the true patterns in the training data but also memorizes the noise or random fluctuations. This leads to high accuracy on the training set but poor
Sources: ['machine_learning.csv | row 2', 'machine_learning.csv | row 36']
Type: normal
Question: What is the LEGB scope rule in Python?
Used rewrite: False
Answer: The LEGB scope rule states that Python resolves variable names in the following order: **Local → Enclosing → Global → Built‑in**.
Sources: ['python_programming_course_guide.docx | full file']
Type: normal
Question: What does A = CR mean in matrix multiplication?
Used rewrite: False
Answer: A = C R means that the matrix A is obtained by multiplying the matrix C by the matrix R. In this factorization C is an m × r matrix (with full column rank r) and R is an r × n matrix (with full row ra
Sources: ['strang_linear

In [21]:
import gradio as gr

def answer_question(question):
    if question.strip() == "":
        return "Please write a question.", "", ""

    result = corrective_rag(question)

    if len(result["sources"]) == 0:
        sources_text = "No sources"
    else:
        sources_text = "\n".join(result["sources"])

    if result["used_rewrite"]:
        info = "Query was rewritten to: " + result["new_question"]
    else:
        info = "No rewrite needed"

    return result["answer"], sources_text, info

demo = gr.Interface(
    fn=answer_question,
    inputs=gr.Textbox(label="Your question", lines=2),
    outputs=[
        gr.Textbox(label="Answer", lines=8),
        gr.Textbox(label="Sources"),
        gr.Textbox(label="Query info")
    ],
    title="Corrective RAG System",
    description="Ask a question about the course documents."
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://76b0f22fbdc998c7de.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
